# Proyecto RappiPlus: de datos a decisiones de negocio

## Introducción

El objetivo de este proyecto es evaluar el desempeño del servicio RappiPlus para apoyar decisiones de negocio basadas en datos.

Se trabaja con múltiples datasets del negocio:

- `rappiplus_orders_raw.csv` → información de pedidos, precios, descuentos y revenue
- `rappiplus_catalog.csv` → costos de productos, categorías y proveedores
- `rappiplus_marketing_spend.csv` → inversión en marketing por canal y país
- `events` / `users` / `user_activity` (SQL) → comportamiento del usuario dentro de la plataforma
- `experiment_checkout_ui.csv` → resultados de un experimento A/B en el checkout

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)
2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)
3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)
4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)
5. 🧪 Validar si los cambios generan impacto (test estadístico)
6. 📊 Comunicar los resultados (dashboard en BI)

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer recomendaciones accionables.

> **Nota sobre alcance de este notebook:** `experiment_checkout_ui.csv` se deja reservado para el Paso 5 (test A/B). Aquí trabajamos exclusivamente los tres archivos que alimentan los Pasos 1 y 2: `orders`, `catalog` y `marketing`.

---

# 🔹 Paso 1: Cargar y validar la calidad de los datos

## 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** familiarizarnos con la estructura de los datasets del negocio antes de analizarlos.

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np

# Opciones de visualización para poder inspeccionar bien las tablas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


**Justificación:** solo necesitamos `pandas` para manipulación tabular y `numpy` para operaciones numéricas (NaN, comparaciones). No se requieren librerías de visualización todavía porque este paso es de diagnóstico, no de análisis exploratorio visual.

In [2]:
# Cargar los tres archivos fuente
orders = pd.read_csv('rappiplus_orders_raw.csv')
catalog = pd.read_csv('rappiplus_catalog.csv')
marketing = pd.read_csv('rappiplus_marketing_spend.csv')

print(f"orders:    {orders.shape[0]:,} filas x {orders.shape[1]} columnas")
print(f"catalog:   {catalog.shape[0]:,} filas x {catalog.shape[1]} columnas")
print(f"marketing: {marketing.shape[0]:,} filas x {marketing.shape[1]} columnas")

orders:    25,100 filas x 12 columnas
catalog:   7 filas x 4 columnas
marketing: 1,620 filas x 5 columnas


In [3]:
# Vista rápida de orders
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [4]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  str    
 1   id_usuario          25100 non-null  str    
 2   fecha_hora_pedido   25100 non-null  str    
 3   pais                24800 non-null  str    
 4   dispositivo         25080 non-null  str    
 5   fuente_referencia   25070 non-null  str    
 6   nombre_producto     25070 non-null  str    
 7   categoria_producto  25020 non-null  str    
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), str(8)
memory usage: 2.3 MB


In [5]:
# Vista rápida de catalog
catalog

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [6]:
catalog.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      str    
 1   categoria_producto  7 non-null      str    
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      str    
dtypes: float64(1), str(3)
memory usage: 356.0 bytes


In [7]:
# Vista rápida de marketing
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [8]:
marketing.info()

<class 'pandas.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   str    
 1   pais        1620 non-null   str    
 2   id_campaña  1620 non-null   str    
 3   canal       1519 non-null   str    
 4   gasto       1620 non-null   float64
dtypes: float64(1), str(4)
memory usage: 63.4 KB


**Observaciones iniciales de la vista rápida:**

- `orders` tiene 25,100 filas — vamos a validar cuántas son realmente pedidos únicos.
- `catalog` es una tabla maestra pequeña (7 productos), sin nulos aparentes: sirve como **fuente de verdad** para nombre de producto → categoría y costo unitario.
- `marketing` tiene una fila por combinación de fecha/país/canal.

Con esta primera vista ya sabemos qué esperar de cada tabla. El siguiente paso es cuantificar la calidad de los datos antes de decidir cómo limpiarlos.

## 1.2 Diagnóstico de calidad de datos

**🎯 Objetivo:** cuantificar problemas (nulos, duplicados, inconsistencias de formato, valores imposibles) en cada dataset **antes** de tocar una sola celda, para tomar decisiones de limpieza basadas en evidencia y no a ciegas.

In [9]:
# --- Diagnóstico: valores nulos por columna ---
print("Nulos en orders:")
print(orders.isna().sum())
print()
print("Nulos en catalog:")
print(catalog.isna().sum())
print()
print("Nulos en marketing:")
print(marketing.isna().sum())

Nulos en orders:
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

Nulos en catalog:
nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64

Nulos en marketing:
fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64


**Justificación:** revisar nulos primero nos dice qué columnas necesitan una estrategia de imputación/exclusión y cuáles están completas. `catalog` está 100% completo (es una tabla de referencia pequeña y controlada). `marketing` solo tiene huecos en `canal`. `orders` es el dataset con más columnas afectadas — lo trataremos con más detalle.

In [10]:
# --- Diagnóstico: duplicados ---
print("Filas 100% duplicadas en orders:", orders.duplicated().sum())
print("id_pedido duplicados en orders:", orders['id_pedido'].duplicated().sum())
print("Filas 100% duplicadas en catalog:", catalog.duplicated().sum())
print("Filas 100% duplicadas en marketing:", marketing.duplicated().sum())

Filas 100% duplicadas en orders: 100
id_pedido duplicados en orders: 100
Filas 100% duplicadas en catalog: 0
Filas 100% duplicadas en marketing: 0


**Hallazgo clave:** hay **100 filas completamente duplicadas** en `orders` (mismo `id_pedido` y mismos valores en todas las columnas). Esto es consistente con un problema típico de ingestión (por ejemplo, un proceso ETL que corrió dos veces sobre el mismo lote). Como son duplicados exactos —no solo mismo id con datos distintos—, la limpieza es directa: eliminar la copia repetida y quedarnos con una sola fila por `id_pedido`.

In [11]:
# --- Diagnóstico: consistencia de valores categóricos ---
print("Valores únicos en orders['pais']:")
print(sorted(orders['pais'].dropna().unique()))
print()
print("Valores únicos en orders['categoria_producto']:")
print(sorted(orders['categoria_producto'].dropna().unique()))
print()
print("Valores únicos en catalog['categoria_producto']:")
print(sorted(catalog['categoria_producto'].unique()))
print()
print("Valores únicos en orders['dispositivo']:", orders['dispositivo'].dropna().unique())
print("Valores únicos en orders['fuente_referencia']:", orders['fuente_referencia'].dropna().unique())
print("Valores únicos en marketing['canal']:", marketing['canal'].dropna().unique())

Valores únicos en orders['pais']:
['Argentina', 'Colombia', 'Mexico', 'argentina', 'colombia', 'mexico']

Valores únicos en orders['categoria_producto']:
['Electronica', 'Hogar', 'Moda']

Valores únicos en catalog['categoria_producto']:
['Electrónica', 'Hogar', 'Moda']

Valores únicos en orders['dispositivo']: <StringArray>
['desktop', 'mobile']
Length: 2, dtype: str
Valores únicos en orders['fuente_referencia']: <StringArray>
['organic', 'paid_search', 'social']
Length: 3, dtype: str
Valores únicos en marketing['canal']: <StringArray>
['organic', 'paid_search', 'social']
Length: 3, dtype: str


**Hallazgos clave:**

1. **`pais` tiene mezcla de mayúsculas/minúsculas**: `'Mexico'` y `'mexico'` se están contando como países *distintos* aunque son el mismo valor. Esto rompe cualquier `groupby('pais')`.
2. **`categoria_producto` tiene una inconsistencia de tildes**: `orders` usa `'Electronica'` (sin tilde) mientras que `catalog` — nuestra fuente de verdad — usa `'Electrónica'` (con tilde). Si no se corrige, un `merge` o `groupby` por categoría duplicará la categoría "Electrónica" en dos variantes.
3. `dispositivo`, `fuente_referencia` y `canal` (en marketing) están limpios en cuanto a valores válidos — su único problema son los nulos ya detectados arriba.

Estas inconsistencias de texto son silenciosas: no generan errores, generan **números de negocio incorrectos** (ej. "Argentina" y "argentina" contados como mercados distintos). Por eso se corrigen antes de cualquier análisis.

In [12]:
# --- Diagnóstico: rangos numéricos y valores imposibles en orders ---
orders[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']].describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [13]:
# Cantidades negativas (imposible vender -2 unidades)
print("Filas con cantidad negativa:")
print(orders[orders['cantidad'] < 0][['id_pedido','nombre_producto','cantidad','precio_unitario','monto_total']])
print()
# Cantidades extremas (outliers de varios órdenes de magnitud)
print("Filas con cantidad >= 1,000 (outliers extremos):", (orders['cantidad'] >= 1000).sum())
print(orders.loc[orders['cantidad'] >= 1000, 'cantidad'].value_counts())

Filas con cantidad negativa:
     id_pedido  nombre_producto  cantidad  precio_unitario  monto_total
266  order_266  Phone-Pro-128GB      -2.0           101.31      -192.62
267  order_267  Phone-Pro-128GB      -1.0            43.50       -38.50
268  order_268  Phone-Pro-128GB      -1.0           497.65      -492.65
269  order_269  Phone-Pro-128GB      -1.0           423.53      -423.53

Filas con cantidad >= 1,000 (outliers extremos): 10
cantidad
10000.0    6
20000.0    4
Name: count, dtype: int64


**Hallazgos clave:**

- **4 filas con `cantidad` negativa** (-1 o -2). Vender una cantidad negativa no existe en el negocio real; lo más probable es que sean **devoluciones registradas por error como pedidos nuevos**, o un bug de captura. No se pueden interpretar como ventas válidas.
- **10 filas con `cantidad` de 10,000 o 20,000 unidades**, siempre en números redondos "sospechosamente perfectos" — el patrón típico de un error de tipeo (dígitos de más, ej. escribir `20000` en vez de `2`). Estas 10 filas por sí solas generan hasta **$8.84 millones** en una sola orden, lo que distorsionaría brutalmente cualquier métrica de revenue si no se tratan.

Ninguno de estos dos problemas se puede "adivinar" con el valor correcto — no hay forma de saber si la cantidad real era 1 o 2. Por eso la estrategia no será *corregir* el número, sino **marcarlo como no confiable y excluirlo de los cálculos financieros**, preservando la fila para no perder trazabilidad de que el pedido existió.

In [14]:
# --- Diagnóstico: ¿la categoría de producto en orders coincide con el catálogo? ---
# (usamos una copia normalizada solo para esta comprobación, sin modificar 'orders' todavía)
tmp = orders.dropna(subset=['nombre_producto', 'categoria_producto']).copy()
tmp['categoria_producto_norm'] = tmp['categoria_producto'].replace({'Electronica': 'Electrónica'})
check = tmp.merge(catalog[['nombre_producto', 'categoria_producto']], on='nombre_producto', suffixes=('_orders', '_catalog'))
mismatches = check[check['categoria_producto_norm'] != check['categoria_producto_catalog']]
print("Inconsistencias producto-categoría tras normalizar tildes:", len(mismatches))

Inconsistencias producto-categoría tras normalizar tildes: 0


**Buena noticia:** una vez corregida la tilde de "Electronica" → "Electrónica", **no hay inconsistencias** entre el producto y su categoría — el catálogo y las órdenes están alineados. Esto confirma que el único problema era de formato de texto, no de datos realmente contradictorios.

## 📋 Resumen del diagnóstico (antes de limpiar)

| Dataset | Problema detectado | Cantidad |
|---|---|---|
| `orders` | Filas 100% duplicadas | 100 |
| `orders` | `pais` con mayúsculas/minúsculas mezcladas | 6 variantes → 3 países reales |
| `orders` | `categoria_producto` sin tilde ("Electronica") | inconsistente vs. catálogo |
| `orders` | Nulos en `pais` | 300 |
| `orders` | Nulos en `dispositivo` | 20 |
| `orders` | Nulos en `fuente_referencia` | 30 |
| `orders` | Nulos en `nombre_producto` | 30 |
| `orders` | Nulos en `categoria_producto` | 80 |
| `orders` | Nulos simultáneos en `cantidad`/`precio_unitario`/`monto_descuento` | 50 |
| `orders` | `cantidad` negativa | 4 |
| `orders` | `cantidad` con outliers extremos (10,000 / 20,000) | 10 |
| `catalog` | — | ninguno |
| `marketing` | Nulos en `canal` | 101 |

Con este mapa completo, ahora sí podemos limpiar cada dataset de forma justificada.

---

# 🔹 Paso 1 (continuación): Limpieza de los datasets

## 1.3 Limpieza de `catalog`

In [15]:
catalog_clean = catalog.copy()

# Quitar espacios en blanco accidentales en texto (buena práctica preventiva,
# aunque el diagnóstico no mostró el problema explícitamente en este dataset)
for col in ['nombre_producto', 'categoria_producto', 'proveedor']:
    catalog_clean[col] = catalog_clean[col].str.strip()

print("catalog_clean listo:", catalog_clean.shape)
catalog_clean

catalog_clean listo: (7, 4)


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


**Justificación:** `catalog` no presentó nulos, duplicados ni inconsistencias — es la tabla más pequeña y controlada. La única acción es una limpieza preventiva de espacios en blanco en texto, una práctica estándar que no altera ningún valor pero evita errores silenciosos en futuros `merge` (un espacio invisible al final de un nombre de producto haría fallar el cruce con `orders`).

## 1.4 Limpieza de `marketing`

In [16]:
marketing_clean = marketing.copy()

# El canal se puede recuperar de forma 100% confiable a partir de id_campaña,
# que siempre sigue el patrón "<canal>_<pais>" (ej. "organic_Mexico" -> "organic")
print("Antes de imputar, nulos en canal:", marketing_clean['canal'].isna().sum())

canal_faltante = marketing_clean['canal'].isna()
marketing_clean.loc[canal_faltante, 'canal'] = (
    marketing_clean.loc[canal_faltante, 'id_campaña']
    .str.extract(r'^([a-z_]+)_[A-Za-z]+$')[0]
)

print("Después de imputar, nulos en canal:", marketing_clean['canal'].isna().sum())
print(marketing_clean['canal'].value_counts())

Antes de imputar, nulos en canal: 101
Después de imputar, nulos en canal: 0
canal
organic        540
paid_search    540
social         540
Name: count, dtype: int64


**Justificación:** a diferencia de una imputación estadística (media, moda), aquí tenemos una **regla determinística**: `id_campaña` ya contiene el canal como prefijo (`organic_Mexico`, `paid_search_Colombia`, etc.). No es una suposición, es dato que ya existe en otra columna — por eso se recupera el 100% de los nulos sin perder información ni inventar valores.

In [17]:
# Validaciones finales de marketing_clean
print("Duplicados:", marketing_clean.duplicated().sum())
print("Gasto negativo:", (marketing_clean['gasto'] < 0).sum())
print("Nulos restantes:")
print(marketing_clean.isna().sum())
print()
print("Cobertura de fechas x país x canal (se espera 9 combinaciones por día):")
print(marketing_clean.groupby('fecha').size().value_counts())

Duplicados:

 0
Gasto negativo: 0
Nulos restantes:
fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64

Cobertura de fechas x país x canal (se espera 9 combinaciones por día):
9    180
Name: count, dtype: int64


**Resultado:** `marketing_clean` queda sin nulos, sin duplicados, sin montos negativos y con las 9 combinaciones esperadas (3 países × 3 canales) en prácticamente todos los días. Este dataset requirió la limpieza más simple de los tres.

## 1.5 Limpieza de `orders`

Esta es la tabla más compleja. La abordamos en sub-pasos, cada uno con su propia justificación, para que el razonamiento quede documentado.

### 1.5.1 Eliminar duplicados exactos

In [18]:
orders_clean = orders.copy()

antes = len(orders_clean)
orders_clean = orders_clean.drop_duplicates(keep='first')
despues = len(orders_clean)

print(f"Filas antes: {antes:,}")
print(f"Filas después de quitar duplicados exactos: {despues:,}")
print(f"Filas eliminadas: {antes - despues}")

Filas antes: 25,100
Filas después de quitar duplicados exactos: 25,000
Filas eliminadas: 100


**Justificación:** al ser duplicados **100% exactos** (mismo `id_pedido` y mismos valores en todas las columnas), no hay ambigüedad sobre cuál conservar — son la misma fila repetida, probablemente por un doble procesamiento en la ingesta. Se conserva la primera aparición y se descarta el resto.

### 1.5.2 Normalizar texto: mayúsculas/minúsculas y tildes

In [19]:
# Normalizar 'pais': mismo país sin importar cómo se escribió
print("Antes:", sorted(orders_clean['pais'].dropna().unique()))
orders_clean['pais'] = orders_clean['pais'].str.strip().str.title()
print("Después:", sorted(orders_clean['pais'].dropna().unique()))

Antes: ['Argentina', 'Colombia', 'Mexico', 'argentina', 'colombia', 'mexico']
Después: ['Argentina', 'Colombia', 'Mexico']


In [20]:
# Normalizar 'categoria_producto': alinear tildes con la fuente de verdad (catalog)
print("Antes:", sorted(orders_clean['categoria_producto'].dropna().unique()))
orders_clean['categoria_producto'] = orders_clean['categoria_producto'].replace({'Electronica': 'Electrónica'})
print("Después:", sorted(orders_clean['categoria_producto'].dropna().unique()))

Antes: ['Electronica', 'Hogar', 'Moda']


Después: ['Electrónica', 'Hogar', 'Moda']


**Justificación:** `.str.title()` unifica `'mexico'`/`'Mexico'` → `'Mexico'` sin necesidad de un diccionario manual, porque los tres países son palabras simples de una sola capitalización. Para la categoría, usamos un mapeo explícito porque es un caso puntual (una sola variante mal escrita) y así garantizamos que el valor final coincide carácter por carácter con `catalog`, que es nuestra fuente de verdad.

### 1.5.3 Recuperar `categoria_producto` faltante desde el catálogo

In [21]:
faltantes_antes = orders_clean['categoria_producto'].isna().sum()

# Diccionario producto -> categoría, construido desde la tabla maestra
mapa_categoria = catalog_clean.set_index('nombre_producto')['categoria_producto']

mask_recuperable = orders_clean['categoria_producto'].isna() & orders_clean['nombre_producto'].notna()
orders_clean.loc[mask_recuperable, 'categoria_producto'] = (
    orders_clean.loc[mask_recuperable, 'nombre_producto'].map(mapa_categoria)
)

faltantes_despues = orders_clean['categoria_producto'].isna().sum()
print(f"Nulos en categoria_producto -> antes: {faltantes_antes}, después: {faltantes_despues}")
print(f"Recuperados vía catálogo: {faltantes_antes - faltantes_despues}")
print(f"Quedan sin recuperar (porque tampoco tienen nombre_producto): {faltantes_despues}")

Nulos en categoria_producto -> antes: 80, después: 30
Recuperados vía catálogo: 50
Quedan sin recuperar (porque tampoco tienen nombre_producto): 30


**Justificación:** `categoria_producto` no es información independiente — es una consecuencia directa de `nombre_producto`, y esa relación 1 a 1 ya existe en `catalog`. Cuando conocemos el producto pero no la categoría, no hace falta imputar con "Desconocido": simplemente la **recuperamos** desde la tabla maestra, que es dato real, no una suposición. Los pocos casos que siguen sin categoría son los mismos que tampoco tienen `nombre_producto` — ahí sí no hay nada de qué recuperarla.

### 1.5.4 Recalcular el monto esperado y clasificar diferencias

Antes de decidir qué hacer con los nulos numéricos y los valores imposibles, construimos columnas de diagnóstico que nos permiten *ver* la magnitud del problema fila por fila, igual que se hizo en la iteración anterior de este proyecto.

In [22]:
# Monto que "debería" dar la operación: cantidad x precio_unitario - descuento
orders_clean['monto_calculado'] = (
    orders_clean['cantidad'] * orders_clean['precio_unitario'] - orders_clean['monto_descuento']
)

# Diferencia contra el monto_total que trae el archivo original
orders_clean['diferencia_monto'] = orders_clean['monto_total'] - orders_clean['monto_calculado']
orders_clean['diferencia_abs'] = orders_clean['diferencia_monto'].abs()
orders_clean['diferencia_redondeada'] = orders_clean['diferencia_monto'].round(2)

# Bandera: la fila no se puede validar porque le faltan campos numéricos de entrada
orders_clean['datos_numericos_incompletos'] = orders_clean[['cantidad', 'precio_unitario', 'monto_descuento']].isna().any(axis=1)

def clasificar_diferencia(row):
    if row['datos_numericos_incompletos']:
        return 'Sin información'
    if row['diferencia_abs'] == 0:
        return 'Sin diferencia'
    if row['diferencia_abs'] <= 0.01:
        return 'Redondeo ±0.01'
    return 'Diferencia relevante'

orders_clean['tipo_diferencia'] = orders_clean.apply(clasificar_diferencia, axis=1)
orders_clean['tipo_diferencia'].value_counts()

tipo_diferencia
Sin diferencia          18378
Redondeo ±0.01           5427
Diferencia relevante     1145
Sin información            50
Name: count, dtype: int64

**Justificación:** en vez de "arreglar" `monto_total` a ciegas, calculamos cuál *debería* ser el monto según sus componentes (`cantidad x precio - descuento`) y lo comparamos contra el valor real del archivo. Esto separa el ruido inofensivo (diferencias de ±$0.01 por redondeo, normales en cualquier sistema de facturación) de las diferencias que sí requieren atención (`"Diferencia relevante"`), y deja explícitamente marcadas las filas que ni siquiera se pueden evaluar por falta de datos (`"Sin información"`).

### 1.5.5 Marcar cantidades inválidas (negativas y outliers extremos)

In [23]:
# a) Cantidades negativas: no existen en un pedido real; lo más probable
#    es que sean devoluciones o errores de captura, no ventas.
cantidad_negativa = orders_clean['cantidad'] < 0

# b) Cantidades extremas: usamos un umbral de negocio (1,000 unidades en un
#    solo pedido de estos productos es fuera de todo rango realista) en vez de
#    un método puramente estadístico, porque el patrón (10,000 / 20,000 exactos)
#    apunta a error de tipeo, no a una compra corporativa legítima.
cantidad_extrema = orders_clean['cantidad'] >= 1000

orders_clean['cantidad_valida'] = ~(cantidad_negativa | cantidad_extrema | orders_clean['cantidad'].isna())

print("Cantidad negativa:", cantidad_negativa.sum())
print("Cantidad extrema (>=1,000):", cantidad_extrema.sum())
print("Total marcado como cantidad NO válida:", (~orders_clean['cantidad_valida']).sum())

Cantidad negativa: 4
Cantidad extrema (>=1,000): 10
Total marcado como cantidad NO válida: 64


**Justificación de la decisión (por qué no corregimos el número):** no existe forma de inferir cuál era la cantidad real detrás de un `-2` o un `20000` — no es un error de fórmula que se pueda recalcular, es un error de captura en el origen. **Corregir un valor sin evidencia sería inventar datos.** La decisión correcta es:

1. **No eliminar la fila** — el pedido existió, tiene un `id_usuario`, una fecha, un producto; borrar la fila completa perdería esa información para análisis de funnel/retención donde la cantidad no es relevante.
2. **Sí excluirla de cualquier cálculo financiero** (revenue, profit, ticket promedio) — porque ahí sí distorsionaría el resultado de forma severa.

Por eso creamos una bandera (`cantidad_valida`) en vez de modificar o borrar datos.

### 1.5.6 Bandera consolidada para análisis financiero

In [24]:
# Una sola bandera que resume si la fila es confiable para sumar en KPIs de revenue/profit:
# requiere que los campos numéricos estén completos Y que la cantidad sea realista.
orders_clean['incluir_en_kpis_financieros'] = (
    (~orders_clean['datos_numericos_incompletos']) & (orders_clean['cantidad_valida'])
)

orders_clean['incluir_en_kpis_financieros'].value_counts()

incluir_en_kpis_financieros
True     24936
False       64
Name: count, dtype: int64

**Justificación:** en vez de dejar que cada análisis futuro (Paso 2: revenue, Paso 3: funnel...) tenga que repetir esta lógica de exclusión, se deja **una sola columna booleana reutilizable**. Cualquier notebook posterior puede hacer `orders_clean[orders_clean['incluir_en_kpis_financieros']]` y trabajar con la certeza de que esas filas son aritméticamente confiables, sin tener que redescubrir el problema.

### 1.5.7 Nulos categóricos restantes (`pais`, `dispositivo`, `fuente_referencia`, `nombre_producto`)

In [25]:
print("Nulos restantes antes de esta etapa:")
print(orders_clean[['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']].isna().sum())

Nulos restantes antes de esta etapa:
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
dtype: int64


In [26]:
# Estos nulos no son recuperables desde ninguna otra columna del propio dataset
# (a diferencia de categoria_producto, que sí se pudo derivar de nombre_producto).
# Se imputan con la etiqueta explícita "Desconocido" en vez de dejarlos como NaN:
# esto evita que un groupby('pais') silenciosamente descarte esas filas,
# y dejamos explícito que el dato faltó, en vez de simularlo con la moda.
for col in ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']:
    orders_clean[col] = orders_clean[col].fillna('Desconocido')

print("Nulos restantes después:")
print(orders_clean[['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']].isna().sum())

Nulos restantes después:
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
dtype: int64


**Justificación:** para variables **categóricas** usadas en agrupaciones (país, dispositivo, canal de referencia), imputar con la moda (el valor más frecuente) *inventaría* un dato — haría parecer que un pedido vino de escritorio cuando en realidad no lo sabemos. Usar `"Desconocido"` como categoría explícita:

- Mantiene la fila disponible para análisis donde esa columna no es la variable de interés (ej. sigue sumando revenue aunque no sepamos el país).
- Es transparente: cualquier reporte mostrará "Desconocido" como una porción visible, en vez de esconder el problema dentro de "Mexico" o "desktop".
- No se aplica a columnas **numéricas** (`cantidad`, `precio_unitario`, etc.) porque ahí ya existe la bandera `incluir_en_kpis_financieros`, que es la forma correcta de tratar esos nulos sin inventar números.

### 1.5.8 Verificación final de tipos y formato de fecha

In [27]:
# Confirmar que fecha_hora_pedido tiene formato consistente y convertirla a datetime real
print("Longitudes de fecha_hora_pedido:", orders_clean['fecha_hora_pedido'].str.len().unique())
orders_clean['fecha_hora_pedido'] = pd.to_datetime(orders_clean['fecha_hora_pedido'])
print("Rango de fechas:", orders_clean['fecha_hora_pedido'].min(), "a", orders_clean['fecha_hora_pedido'].max())
print("dtype:", orders_clean['fecha_hora_pedido'].dtype)

Longitudes de fecha_hora_pedido: [10]
Rango de fechas: 2025-01-01 00:00:00 a 2025-06-30 00:00:00
dtype: datetime64[us]


**Justificación:** todas las fechas ya venían en formato `YYYY-MM-DD` consistente (10 caracteres, sin excepciones), así que la conversión a `datetime` es segura y no requiere parsing especial. Trabajar con un tipo `datetime` real (en vez de texto) es lo que permite después operaciones de calendario para el análisis de retención por cohortes (Paso 4).

## 1.6 Validación final de `orders_clean`

In [28]:
print("Forma final:", orders_clean.shape)
print()
print("Nulos restantes (deben ser 0 en todas las columnas de identidad/categóricas):")
print(orders_clean.isna().sum())
print()
print("Duplicados restantes:", orders_clean.duplicated().sum())
print("id_pedido únicos:", orders_clean['id_pedido'].is_unique)
print()
print("Distribución de calidad para KPIs financieros:")
print(orders_clean['incluir_en_kpis_financieros'].value_counts(normalize=True).round(4) * 100)
orders_clean.head()

Forma final: (25000, 20)

Nulos restantes (deben ser 0 en todas las columnas de identidad/categóricas):
id_pedido                       0
id_usuario                      0
fecha_hora_pedido               0
pais                            0
dispositivo                     0
fuente_referencia               0
nombre_producto                 0
categoria_producto              0
cantidad                       50
precio_unitario                50
monto_descuento                50
monto_total                     0
monto_calculado                50
diferencia_monto               50
diferencia_abs                 50
diferencia_redondeada          50
datos_numericos_incompletos     0
tipo_diferencia                 0
cantidad_valida                 0
incluir_en_kpis_financieros     0
dtype: int64

Duplicados restantes: 0
id_pedido únicos: True

Distribución de calidad para KPIs financieros:
incluir_en_kpis_financieros
True     99.74
False     0.26
Name: proportion, dtype: float64


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_calculado,diferencia_monto,diferencia_abs,diferencia_redondeada,datos_numericos_incompletos,tipo_diferencia,cantidad_valida,incluir_en_kpis_financieros
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,665.38,-0.01,0.01,-0.01,False,Redondeo ±0.01,True,True
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electrónica,1.0,176.86,5.0,171.86,171.86,0.00,0.00,0.00,False,Sin diferencia,True,True
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,195.98,0.01,0.01,0.01,False,Diferencia relevante,True,True
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electrónica,1.0,257.87,15.0,242.87,242.87,0.00,0.00,0.00,False,Sin diferencia,True,True
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,336.28,0.00,0.00,0.00,False,Sin diferencia,True,True


**Nota sobre los nulos numéricos que persisten:** las columnas `cantidad`, `precio_unitario`, `monto_descuento`, `monto_calculado`, `diferencia_monto`, `diferencia_abs` y `diferencia_redondeada` conservarán NaN en las filas marcadas como `datos_numericos_incompletos = True`. Esto es intencional: no se imputan porque no hay ninguna base razonable para inventar un precio o una cantidad, y la columna `incluir_en_kpis_financieros` ya indica que deben excluirse de sumas y promedios. `monto_total` sí está completo en el 100% de las filas (incluidas esas), por lo que sigue siendo utilizable para métricas de ingresos incluso cuando no se puede *validar* su desglose interno.

## 1.7 Guardar los archivos limpios

**🎯 Objetivo:** dejar los tres datasets listos, documentados y reproducibles para los siguientes pasos del proyecto (Paso 2: revenue y profit).

In [29]:
orders_clean.to_csv('rappiplus_orders_clean.csv', index=False)
catalog_clean.to_csv('rappiplus_catalog_clean.csv', index=False)
marketing_clean.to_csv('rappiplus_marketing_clean.csv', index=False)

print("Archivos guardados:")
print(" - rappiplus_orders_clean.csv   ->", orders_clean.shape)
print(" - rappiplus_catalog_clean.csv  ->", catalog_clean.shape)
print(" - rappiplus_marketing_clean.csv->", marketing_clean.shape)

Archivos guardados:
 - rappiplus_orders_clean.csv   -> (25000, 20)
 - rappiplus_catalog_clean.csv  -> (7, 4)
 - rappiplus_marketing_clean.csv-> (1620, 5)


## 📋 Resumen de decisiones de limpieza

| # | Problema | Dataset | Decisión | Justificación breve |
|---|---|---|---|---|
| 1 | 100 filas duplicadas exactas | orders | Eliminar (`drop_duplicates`) | Copia idéntica, sin ambigüedad sobre cuál conservar |
| 2 | `pais` con mayúsculas/minúsculas mezcladas | orders | Normalizar con `.str.title()` | Mismo país, distinto formato de texto |
| 3 | `categoria_producto` sin tilde ("Electronica") | orders | Mapear a `"Electrónica"` según catálogo | El catálogo es la fuente de verdad |
| 4 | `categoria_producto` nula (80) | orders | Recuperar 50 vía `nombre_producto` → catálogo; 30 quedan `"Desconocido"` | Relación 1:1 real, no es una suposición |
| 5 | Nulos en `pais`, `dispositivo`, `fuente_referencia`, `nombre_producto` | orders | Imputar con `"Desconocido"` | No inventar un valor; mantener la fila útil para otras columnas |
| 6 | Nulos simultáneos en `cantidad`/`precio_unitario`/`monto_descuento` (50 filas) | orders | Dejar como NaN + bandera `datos_numericos_incompletos` | No hay base para inventar montos; se excluyen de KPIs financieros |
| 7 | `cantidad` negativa (4 filas) | orders | Mantener fila, bandera `cantidad_valida = False` | Posible devolución/error de captura; no es una venta real |
| 8 | `cantidad` extrema (10,000–20,000; 10 filas) | orders | Mantener fila, bandera `cantidad_valida = False` | Error de tipeo evidente; imposible inferir el valor correcto |
| 9 | Canal nulo en marketing (101 filas) | marketing | Extraer de `id_campaña` | Dato ya existente en otra columna, recuperación 100% confiable |
| 10 | Formato de fecha | orders | Convertir a `datetime` | Necesario para análisis de cohortes en pasos posteriores |

## ➡️ Próximos pasos

Con `rappiplus_orders_clean.csv`, `rappiplus_catalog_clean.csv` y `rappiplus_marketing_clean.csv` listos y documentados, el proyecto continúa en el **Paso 2: Analizar si el negocio es rentable (revenue, costos y profit)**, cruzando `orders_clean` con `catalog_clean` para calcular margen por producto/pedido, y con `marketing_clean` para evaluar retorno de la inversión publicitaria por canal y país.

`experiment_checkout_ui.csv` permanece sin tocar hasta el **Paso 5**, donde se usará para el test A/B del checkout.